# Creación BD S&P500

La idea es que del dataset con los precios, se tome el último día y se resten 1000 días, después se compare con el otro dataset de tickers para evaluar el sesgo de supervivencia. Los enlaces a dichos datasets se encuentran a continuación:

Dataset con la información de precios: https://www.kaggle.com/datasets/camnugent/sandp500?resource=download

Dataset con la información de tickers: https://github.com/fja05680/sp500

In [3]:
import pandas as pd
import os

# ---------------------------------------------------------
# BLOQUE 1: Configuración de Rutas y Carga Inicial
# ---------------------------------------------------------
folder_path = r'../data/raw'
path_main = os.path.join(folder_path, 'all_stocks_5yr.csv')
path_components = os.path.join(folder_path, 'S&P 500 Historical Components & Changes(01-17-2026).csv')

print("Cargando dataset principal...")
df = pd.read_csv(path_main)
df['date'] = pd.to_datetime(df['date'])

# Aseguramos el orden cronológico para que el recorte de registros sea correcto
df = df.sort_values(['Name', 'date'])

# ---------------------------------------------------------
# BLOQUE 2: Filtrado por Registros (Metodología del Paper)
# ---------------------------------------------------------
# El paper utiliza 1000 días de trading (750 training + 250 trading).
# Filtramos primero para no pasarnos de febrero de 2018
df_2018 = df[df['date'].dt.year <= 2018].copy()

print("Filtrando los últimos 1000 registros de trading por cada acción...")
# Usamos groupby + tail(1000) para obtener los últimos 1000 registros de cada ticker
df_filtrado = df_2018.groupby('Name').tail(1000).copy()

# Verificación de la fecha máxima y mínima resultante
print(f"Rango de fechas en el dataset filtrado: {df_filtrado['date'].min().date()} a {df_filtrado['date'].max().date()}")

Cargando dataset principal...


FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/all_stocks_5yr.csv'

In [17]:
# ---------------------------------------------------------
# BLOQUE 3: Análisis de Sesgo de Supervivencia
# ---------------------------------------------------------
print("\nAnalizando componentes históricos para verificar sesgo...")
df_hist_comp = pd.read_csv(path_components)
df_hist_comp['date'] = pd.to_datetime(df_hist_comp['date'])

# Obtenemos la lista oficial del S&P 500 en la fecha final del dataset [cite: 1086]
fecha_final = df_filtrado['date'].max()
comp_en_fecha = df_hist_comp[df_hist_comp['date'] <= fecha_final].sort_values('date', ascending=False).iloc[0]

tickers_maestros = set(comp_en_fecha['tickers'].split(','))
tickers_en_dataset = set(df_filtrado['Name'].unique())

missing_stocks = tickers_maestros - tickers_en_dataset
survivors_only = tickers_en_dataset - tickers_maestros

print(f"--- RESULTADOS DEL ANÁLISIS ---")
print(f"Empresas en la Lista Maestra (Oficial): {len(tickers_maestros)}")
print(f"Empresas en tu Dataset: {len(tickers_en_dataset)}")
print(f"Empresas faltantes (Sesgo de Supervivencia): {len(missing_stocks)}")
print(f"Empresas 'extra' (Posibles deslistadas incluidas): {len(survivors_only)}")


Analizando componentes históricos para verificar sesgo...
--- RESULTADOS DEL ANÁLISIS ---
Empresas en la Lista Maestra (Oficial): 506
Empresas en tu Dataset: 505
Empresas faltantes (Sesgo de Supervivencia): 7
Empresas 'extra' (Posibles deslistadas incluidas): 6


In [18]:
# ---------------------------------------------------------
# BLOQUE 4: Almacenamiento y Verificación Final
# ---------------------------------------------------------
output_path = os.path.join(folder_path, 'sp500_ready_1000_records.csv')
df_filtrado.to_csv(output_path, index=False)

# Verificamos un ticker al azar para confirmar que tiene (o se acerca a) los 1000 registros
ejemplo_ticker = df_filtrado['Name'].iloc[0]
conteo_ejemplo = len(df_filtrado[df_filtrado['Name'] == ejemplo_ticker])
print(f"\nVerificación: La acción {ejemplo_ticker} tiene {conteo_ejemplo} registros.")
print(f"Archivo final guardado en: {output_path}")


Verificación: La acción A tiene 1000 registros.
Archivo final guardado en: C:\Users\USUARIO\Downloads\sp500_ready_1000_records.csv


In [19]:
# ---------------------------------------------------------
# BLOQUE 5: Identificación de Empresas por Volumen de Datos
# ---------------------------------------------------------

# 1. Contamos cuántos registros tiene cada empresa (usando el dataframe de 2018)
conteo_registros = df_2018['Name'].value_counts()

# 2. Filtramos las empresas que cumplen el requisito de 1000 registros
empresas_completas = conteo_registros[conteo_registros >= 1000]
empresas_incompletas = conteo_registros[conteo_registros < 1000]

# 3. Mostramos los resultados estadísticos
print(f"--- ANÁLISIS DE DENSIDAD DE DATOS ---")
print(f"Total de empresas analizadas: {len(conteo_registros)}")
print(f"Empresas con >= 1000 registros: {len(empresas_completas)} (Aptas para replicación total)")
print(f"Empresas con < 1000 registros: {len(empresas_incompletas)} (Historia parcial)")

# 4. Ver ejemplos de empresas que no llegan a los 1000 registros
if len(empresas_incompletas) > 0:
    print(f"\nEjemplos de empresas con pocos datos (menos de 1000):")
    # Mostramos las 5 empresas con menos registros
    print(empresas_incompletas.tail(5))

# 5. Opcional: Crear una lista de tickers "aptos" para el entrenamiento intensivo
tickers_aptos = empresas_completas.index.tolist()

--- ANÁLISIS DE DENSIDAD DE DATOS ---
Total de empresas analizadas: 505
Empresas con >= 1000 registros: 483 (Aptas para replicación total)
Empresas con < 1000 registros: 22 (Historia parcial)

Ejemplos de empresas con pocos datos (menos de 1000):
Name
DXC     215
BHGE    152
BHF     143
DWDP    109
APTV     44
Name: count, dtype: int64
